# Phase 5 (LSI) - Model 4 (GQA + MHA mix, MoE) + LoRMA
Multi-label statute identification with the frozen Phase-1 Model 4 backbone (GQA and full-MHA layers, Mixture-of-Experts feed-forward).
LoRMA is written by hand (no peft). Head: BiGRU over chunk vectors + attention pooling. Same data, folds, chunking (32 chunks, training = inference), loss and thresholds as the Model 1 notebooks, so the out-of-fold predictions line up for an ensemble.

**Attach (Kaggle "Add Input"):**
1. The LSI data-prep output (contains `tokenized_lsi.pkl`, `lsi_utils.py`)
2. The output of your Model 4 Phase-1 notebook: the folder `legal-gqamoe-final` (`config.json` + weights) **and** the three code files it wrote next to it: `model.py`, `moe.py`, `legal_gqa_model.py` (if they are not in that output, put the three files in one small dataset)

**Output:** `/kaggle/working/lsi_m4_lorma/` (checkpoints, OOF probabilities, test probabilities, submission)

## 1. Configuration

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import json
import glob
import math
import time
import random
import shutil

import numpy as np

# Env vars are only for local testing. On Kaggle leave them alone.
DATA_DIR = os.environ.get("LSI_DATA_DIR", None)
MODEL_DIR = os.environ.get("LSI_MODEL_DIR", None)
MODEL_PY = os.environ.get("LSI_MODEL_PY", None)      # model.py (moe.py and legal_gqa_model.py sit next to it)
INPUT_ROOT = os.environ.get("LSI_INPUT_ROOT", "/kaggle/input")
OUT_DIR = os.environ.get("LSI_OUT_DIR", "/kaggle/working/lsi_m4_lorma")

MODEL_TYPE = "legal_gqa_mha_moe"  # model_type inside the Phase-1 Model 4 config.json

SMOKE_TEST = False                # True = tiny run to check the code end to end
SMOKE_DOCS = 64
SMOKE_EPOCHS = 1

SEED = 42
RUN_FOLDS = [0]                   # start with fold 0; add 1..4 later if time allows
N_FOLDS = 5

# Chunking (must match the data-prep notebook)
CHUNK_LEN = 1024
CHUNK_OVERLAP = 128
MAX_CHUNKS_TRAIN = 32             # same chunking as inference
MAX_CHUNKS_EVAL = 32              # all chunks at inference (covers about 99.8% of docs)
CHUNK_PICK = "uniform"            # chunks spread over the whole document (training and inference)

# LoRMA (hand-written)
LORMA_R = 16
LORMA_ALPHA = 16
LORMA_DROPOUT = 0.05
# Same 7 layer names as the Model 4 Phase-1 / Phase-3 runs: attention q,k,v,o and the feed-forward gate, up, down
# (in the MoE blocks these are inside every expert). The router is not adapted.
LORMA_TARGET_NAMES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Head and training
HEAD_HIDDEN = 256                 # GRU size per direction
LR = 5e-4
WEIGHT_DECAY = 0.01
EPOCHS = 2
WARMUP_FRAC = 0.05
MICRO_BATCH = 2                   # documents per forward pass
GRAD_ACC = 8                      # effective batch = 16 documents
MICRO_CHUNKS = 4                  # chunks per backbone forward pass
GRAD_CLIP = 1.0
POS_WEIGHT_MODE = "sqrt"          # "sqrt" damps the very large pos_weights of rare statutes
RARE_OVERSAMPLE = True           # sample documents with rare statutes more often (with replacement)
RARE_OVERSAMPLE_CAP = 10.0       # max weight of one document, so one doc is not repeated too much
THRESH_MIN_POS = 20              # per-label thresholds only for statutes with at least this many positives
EVAL_BATCH = 4

# Session time cap (Kaggle GPU sessions are about 12 hours). The run saves and stops, and resumes on the next run.
SESSION_CAP_HOURS = 10.5

# Resume / predict-only. Kaggle starts every run with an empty working folder, so attach the OUTPUT of the earlier run
# (or a Dataset made from it) as an input: its checkpoints are copied back automatically.
RESUME_DIR = None                 # None = search the inputs for a folder with the same name as OUT_DIR
EVAL_ONLY = False                 # True = no training: load the saved checkpoint and only predict validation + test

if SMOKE_TEST:
    print("SMOKE_TEST is on: tiny data, 1 epoch, to check the code only.")
print("Config loaded.")

## 2. Imports and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint

START_TIME = time.time()
SESSION_CAP_SEC = SESSION_CAP_HOURS * 3600

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    if torch.cuda.is_bf16_supported():
        AMP_DTYPE = torch.bfloat16
        USE_SCALER = False
    else:
        AMP_DTYPE = torch.float16
        USE_SCALER = True
else:
    AMP_DTYPE = torch.float32
    USE_SCALER = False
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE, "| amp dtype:", AMP_DTYPE, "| grad scaler:", USE_SCALER)

os.makedirs(OUT_DIR, exist_ok=True)


def copy_missing_files(src_dir, dst_dir):
    copied = []
    for name in sorted(os.listdir(src_dir)):
        src_path = os.path.join(src_dir, name)
        dst_path = os.path.join(dst_dir, name)
        if os.path.isfile(src_path) and not os.path.exists(dst_path):
            shutil.copy(src_path, dst_path)
            copied.append(name)
    return copied


def find_resume_dir():
    # A folder with the same name as OUT_DIR that holds checkpoints (ckpt_fold*.pt).
    if RESUME_DIR is not None:
        return RESUME_DIR
    wanted = os.path.basename(os.path.normpath(OUT_DIR))
    root = os.environ.get("LSI_RESUME_ROOT", os.environ.get("LSI_INPUT_ROOT", "/kaggle/input"))
    folders = []
    for dir_path, dir_names, file_names in os.walk(root):
        if os.path.basename(dir_path) != wanted:
            continue
        has_ckpt = False
        for file_name in file_names:
            if file_name.startswith("ckpt_fold") and file_name.endswith(".pt"):
                has_ckpt = True
        if has_ckpt and os.path.abspath(dir_path) != os.path.abspath(OUT_DIR):
            folders.append(dir_path)
    if len(folders) == 0:
        return None
    return sorted(folders)[-1]


resume_src = find_resume_dir()
if resume_src is None:
    print("No earlier run found in the inputs: starting fresh.")
else:
    copied_files = copy_missing_files(resume_src, OUT_DIR)
    print("Resumed from:", resume_src, "| files copied:", copied_files)

## 3. Find the inputs

In [ ]:
def find_one_file(search_roots, file_name):
    for root in search_roots:
        for dir_path, dir_names, file_names in os.walk(root):
            if file_name in file_names:
                return os.path.join(dir_path, file_name)
    return None


def find_files(file_name):
    return sorted(glob.glob(os.path.join(INPUT_ROOT, "**", file_name), recursive=True))


def find_model_dir():
    # The Phase-1 Model 4 folder: config.json with model_type legal_gqa_mha_moe, plus weights.
    found = []
    for cfg_path in find_files("config.json"):
        folder = os.path.dirname(cfg_path)
        try:
            with open(cfg_path) as fh:
                model_type = json.load(fh).get("model_type")
        except Exception:
            continue
        has_weights = os.path.exists(os.path.join(folder, "model.safetensors")) or \
                      os.path.exists(os.path.join(folder, "pytorch_model.bin"))
        if model_type == MODEL_TYPE and has_weights:
            found.append(folder)
    if len(found) == 0:
        return None
    final_ones = []
    for folder in found:
        if "checkpoint-" not in folder:
            final_ones.append(folder)
    if len(final_ones) > 0:
        return final_ones[0]
    return max(found, key=lambda d: int(d.rsplit("-", 1)[1]))


def find_model_py(model_dir):
    # model.py must have moe.py in the same folder.
    places = ["/kaggle/working/model.py", os.path.join(model_dir, "model.py"),
              os.path.join(os.path.dirname(model_dir), "model.py")]
    for moe_path in find_files("moe.py"):
        places.append(os.path.join(os.path.dirname(moe_path), "model.py"))
    for p in places:
        folder = os.path.dirname(p)
        if os.path.exists(p) and os.path.exists(os.path.join(folder, "moe.py")):
            return p
    return None


if DATA_DIR is None:
    pkl_path = find_one_file([INPUT_ROOT, "/kaggle/working"], "tokenized_lsi.pkl")
    if pkl_path is None:
        raise FileNotFoundError("tokenized_lsi.pkl not found. Attach the LSI data-prep output.")
    DATA_DIR = os.path.dirname(pkl_path)

if MODEL_DIR is None:
    MODEL_DIR = find_model_dir()
    if MODEL_DIR is None:
        raise FileNotFoundError("No Phase-1 Model 4 folder (model_type legal_gqa_mha_moe) found. Attach the Model 4 Phase-1 output.")

if MODEL_PY is None:
    MODEL_PY = find_model_py(MODEL_DIR)
    if MODEL_PY is None:
        raise FileNotFoundError("model.py + moe.py (Model 4 code) not found in one folder. Attach the Model 4 Phase-1 output "
                                "or a small dataset with model.py, moe.py and legal_gqa_model.py.")

print("data dir:", DATA_DIR)
print("model dir:", MODEL_DIR)
print("model file:", MODEL_PY)

## 4. Load the data bundle from the data-prep notebook

In [ ]:
import pickle

with open(os.path.join(DATA_DIR, "tokenized_lsi.pkl"), "rb") as f:
    DATA = pickle.load(f)

if os.path.exists(os.path.join(DATA_DIR, "lsi_utils.py")):
    shutil.copy(os.path.join(DATA_DIR, "lsi_utils.py"), "lsi_utils.py")
sys.path.insert(0, os.getcwd())
import lsi_utils

SEP_ID = int(DATA["sep_id"])
PAD_ID = int(DATA["pad_id"])
STATUTES = [int(s) for s in DATA["statute_ids"]]
N_STATUTES = len(STATUTES)
print("SEP:", SEP_ID, "| PAD:", PAD_ID, "| statutes:", N_STATUTES)

# Pool = train + dev (test has no labels)
pool_ids, pool_labels, pool_fold, pool_doc_ids = [], [], [], []
for split in ["train", "dev"]:
    block = DATA["splits"][split]
    for i in range(len(block["doc_ids"])):
        pool_ids.append(block["ids"][i])
        pool_labels.append(block["labels"][i])
        pool_fold.append(int(block["fold"][i]))
        pool_doc_ids.append(str(block["doc_ids"][i]))

test_block = DATA["splits"]["test"]
test_ids = list(test_block["ids"])
test_doc_ids = [str(x) for x in test_block["doc_ids"]]

Y_POOL = lsi_utils.multi_hot(pool_labels, STATUTES)
N_POOL = len(pool_ids)

if SMOKE_TEST:
    keep_pool = list(range(min(SMOKE_DOCS, N_POOL)))
    pool_ids = [pool_ids[i] for i in keep_pool]
    pool_labels = [pool_labels[i] for i in keep_pool]
    pool_fold = [pool_fold[i] for i in keep_pool]
    pool_doc_ids = [pool_doc_ids[i] for i in keep_pool]
    Y_POOL = Y_POOL[keep_pool]
    N_POOL = len(pool_ids)
    test_ids = test_ids[:SMOKE_DOCS]
    test_doc_ids = test_doc_ids[:SMOKE_DOCS]

# Document sampling weights for rare-statute oversampling.
# A statute's rarity = sqrt(most common count / its count). A document gets the largest rarity of its statutes.
POOL_COUNTS = Y_POOL.sum(axis=0).astype(np.float64)
RARITY = np.sqrt(POOL_COUNTS.max() / np.maximum(POOL_COUNTS, 1.0))
DOC_WEIGHT = np.ones(N_POOL, dtype=np.float64)
for i in range(N_POOL):
    pos_idx = np.where(Y_POOL[i] == 1)[0]
    if len(pos_idx) > 0:
        DOC_WEIGHT[i] = float(RARITY[pos_idx].max())
DOC_WEIGHT = np.minimum(DOC_WEIGHT, RARE_OVERSAMPLE_CAP)
print("doc weight (min / mean / max):", round(float(DOC_WEIGHT.min()), 2),
      round(float(DOC_WEIGHT.mean()), 2), round(float(DOC_WEIGHT.max()), 2))

print("pool docs:", N_POOL, "| test docs:", len(test_ids))
print("fold counts:", {f: pool_fold.count(f) for f in sorted(set(pool_fold))})
print("label cardinality per pool doc (mean):", round(float(Y_POOL.sum(axis=1).mean()), 2))

## 5. Load the frozen Phase-1 Model 4 backbone

In [ ]:
# model.py imports moe.py and legal_gqa_model.py, so their folders must be importable.
sys.path.insert(0, os.path.dirname(MODEL_PY))
if not os.path.exists(os.path.join(os.path.dirname(MODEL_PY), "legal_gqa_model.py")):
    others = find_files("legal_gqa_model.py")
    if len(others) == 0:
        raise FileNotFoundError("legal_gqa_model.py (imported by model.py) not found.")
    sys.path.insert(0, os.path.dirname(others[0]))
    print("legal_gqa_model.py from:", others[0])
from model import LegalGQAMoEForCausalLM

LM, load_info = LegalGQAMoEForCausalLM.from_pretrained(MODEL_DIR, output_loading_info=True)
missing_keys = []
for key in load_info.get("missing_keys", []):
    if key != "lm_head.weight":          # tied to the token embedding
        missing_keys.append(key)
unexpected_keys = list(load_info.get("unexpected_keys", []))
print("Phase-1 weights loaded | missing keys:", len(missing_keys), "| unexpected keys:", len(unexpected_keys))
if len(missing_keys) > 0:
    raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing_keys[:5]))
if LM.config.vocab_size <= SEP_ID:
    raise RuntimeError("tokenizer / model vocab mismatch")
LM.config.use_cache = False
LM.float()                              # experts and adapters in one dtype; autocast handles the speed
LM = LM.to(DEVICE)
BACKBONE = LM


def find_input_embedding(lm):
    # The largest nn.Embedding is the token embedding table.
    best_name = None
    best_module = None
    for name, module in lm.named_modules():
        if isinstance(module, nn.Embedding):
            if best_module is None or module.num_embeddings > best_module.num_embeddings:
                best_name = name
                best_module = module
    return best_name, best_module


def set_submodule(root, dotted_name, new_module):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


EMB_NAME, old_emb = find_input_embedding(BACKBONE)
print("token embedding:", EMB_NAME, "| rows:", old_emb.num_embeddings)
assert EMB_NAME == "wte", "backbone_hidden() reads LM.wte, but the token embedding is named " + str(EMB_NAME)

# PAD id is one past the tokenizer vocabulary, so the embedding table needs one extra row.
if PAD_ID >= old_emb.num_embeddings:
    new_emb = nn.Embedding(PAD_ID + 1, old_emb.embedding_dim).to(old_emb.weight.device, old_emb.weight.dtype)
    with torch.no_grad():
        new_emb.weight[: old_emb.num_embeddings] = old_emb.weight
        new_emb.weight[PAD_ID:] = 0.0      # the PAD row is never used as a real token
    new_emb.weight.requires_grad = False
    set_submodule(BACKBONE, EMB_NAME, new_emb)
    print("embedding table grown from", old_emb.num_embeddings, "to", PAD_ID + 1, "rows (PAD row added)")
else:
    print("PAD id already inside the embedding table:", PAD_ID)

for p in BACKBONE.parameters():
    p.requires_grad = False
BACKBONE.eval()

n_frozen = sum(p.numel() for p in BACKBONE.parameters())
print("backbone:", type(BACKBONE).__name__, "| frozen params:", f"{n_frozen:,}")

# The transformer blocks: the ModuleList named blocks.
BLOCK_LIST_NAME = None
BLOCK_LIST = None
for name, module in BACKBONE.named_modules():
    if isinstance(module, nn.ModuleList) and name.split(".")[-1] in ["blocks", "layers"]:
        BLOCK_LIST_NAME = name
        BLOCK_LIST = module
        break
if BLOCK_LIST is None:
    raise RuntimeError("No ModuleList named blocks or layers found. Check the Model 4 module names.")
print("transformer blocks:", BLOCK_LIST_NAME, "| count:", len(BLOCK_LIST))

# LoRMA targets: linear layers inside the blocks whose name is in LORMA_TARGET_NAMES.
# Everything else (for example the MoE router) stays frozen and is listed so nothing is adapted by accident.
LORMA_TARGETS = []
SKIPPED_LINEARS = {}
for name, module in BACKBONE.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    if not name.startswith(BLOCK_LIST_NAME + "."):
        continue
    last = name.split(".")[-1]
    if last in LORMA_TARGET_NAMES:
        LORMA_TARGETS.append(name)
    else:
        SKIPPED_LINEARS[last] = SKIPPED_LINEARS.get(last, 0) + 1
if len(LORMA_TARGETS) == 0:
    raise RuntimeError("No LoRMA targets found. Check LORMA_TARGET_NAMES against the Model 4 layer names.")
print("LoRMA target linear layers:", len(LORMA_TARGETS))
print("first few:", LORMA_TARGETS[:6])
print("linear layers inside the blocks that stay frozen (name: count):", SKIPPED_LINEARS)


# Hidden states: the same computation as the Phase-1 forward(), stopped before lm_head, so the
# 50k-way vocabulary projection is never computed. The model is causal, so right padding cannot change
# the outputs of the real tokens and no attention mask is needed. Final layer norm is applied.
def backbone_hidden(input_ids, attention_mask):
    T = input_ids.shape[1]
    x = LM.drop(LM.wte(input_ids))
    cos, sin = LM._rope_slice(0, T, input_ids.device, x.dtype)
    for blk in LM.blocks:
        x = blk(x, cos, sin, None, True, None, False)[0]
    return LM.ln_f(x)


HIDDEN = None
with torch.no_grad():
    probe_ids = torch.tensor([[1, 2, 3, SEP_ID]], device=DEVICE)
    probe_mask = torch.ones_like(probe_ids)
    probe_h = backbone_hidden(probe_ids, probe_mask)
    HIDDEN = int(probe_h.shape[-1])
print("backbone hidden size:", HIDDEN)

## 6. Hand-written LoRMA layer (plus variant)
LoRMA (plus): `y = z + (alpha / r) * B (A z)` with `z = W0 x`, i.e. `W = ((alpha / r) B A + I) W0`. `W0` is frozen, `B` starts at zero, so training starts exactly at the Phase-1 weights.

In [ ]:
class LoRMALinear(nn.Module):
    # LoRMA "plus": W = ((alpha / r) * B A + I) W0, with W0 frozen.
    # Output form: y = z + (alpha / r) * B (A z), where z = W0 x (the frozen layer's output).
    # B starts at zero, so training starts exactly at the Phase-1 weights.
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_out, d_in = base.weight.shape
        self.r = r
        self.scale = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, d_out, device=base.weight.device, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r, device=base.weight.device, dtype=torch.float32))
        self.drop = nn.Dropout(dropout)
        self.calls = 0                  # how often this layer ran (used by the activity check)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        with torch.no_grad():
            self.lora_B.zero_()

    def forward(self, x):
        self.calls += 1
        z = self.base(x)
        zf = z.float()
        delta = self.drop(zf) @ self.lora_A.t() @ self.lora_B.t()
        return z + (delta * self.scale).to(z.dtype)


def inject_lorma(backbone):
    lora_modules = []
    for name in LORMA_TARGETS:
        target = backbone.get_submodule(name)
        wrapped = LoRMALinear(target, LORMA_R, LORMA_ALPHA, LORMA_DROPOUT)
        set_submodule(backbone, name, wrapped)
        lora_modules.append(wrapped)
    return lora_modules


# Identity check: LoRMA at init must not change the Phase-1 hidden states.
check_ids = torch.tensor([[5, 9, 12, 40, 77, SEP_ID]], device=DEVICE)
check_mask = torch.ones_like(check_ids)
with torch.no_grad():
    before = backbone_hidden(check_ids, check_mask).float().clone()

LORMA_MODULES = inject_lorma(BACKBONE)
BACKBONE.eval()
for m in LORMA_MODULES:
    m.train()   # keep LoRMA dropout on during training only

with torch.no_grad():
    after = backbone_hidden(check_ids, check_mask).float()
identity_diff = float((before - after).abs().max())
print("LoRMA modules:", len(LORMA_MODULES))
print("identity check (max abs diff at init, should be 0):", identity_diff)
assert identity_diff < 1e-4, "LoRMA changed the Phase-1 output at init"

# Activity check 1: every kind of adapted layer must really run in a forward pass.
# A Mixture-of-Experts layer that read expert weights directly would silently skip the adapters.
for m in LORMA_MODULES:
    m.calls = 0
with torch.no_grad():
    act_ids = torch.randint(0, old_emb.num_embeddings, (4, 256), device=DEVICE)
    backbone_hidden(act_ids, torch.ones_like(act_ids))
calls_by_kind = {}
for name, m in zip(LORMA_TARGETS, LORMA_MODULES):
    kind = name.split(".")[-1]
    if kind not in calls_by_kind:
        calls_by_kind[kind] = [0, 0]
    calls_by_kind[kind][0] += 1
    if m.calls > 0:
        calls_by_kind[kind][1] += 1
for kind in calls_by_kind:
    print("  ", kind.ljust(10), "wrapped:", calls_by_kind[kind][0], "| used in the probe:", calls_by_kind[kind][1])
    assert calls_by_kind[kind][1] > 0, "adapters on " + kind + " are never used in the forward pass"
for m in LORMA_MODULES:
    m.calls = 0

# Activity check 2: with random B the output must change, so the adapters really act on the output.
saved_B = []
for m in LORMA_MODULES:
    saved_B.append(m.lora_B.detach().clone())
with torch.no_grad():
    for m in LORMA_MODULES:
        m.lora_B.normal_(0.0, 0.05)
    perturbed = backbone_hidden(check_ids, check_mask).float()
    for m, b in zip(LORMA_MODULES, saved_B):
        m.lora_B.copy_(b)
sensitivity = float((perturbed - before).abs().max())
print("sensitivity check (output change with random B, must be > 0):", sensitivity)
assert sensitivity > 1e-6, "random adapter weights did not change the output"

LORMA_PARAMS = []
for m in LORMA_MODULES:
    LORMA_PARAMS.append(m.lora_A)
    LORMA_PARAMS.append(m.lora_B)
n_lora = sum(p.numel() for p in LORMA_PARAMS)
print("trainable LoRMA params:", f"{n_lora:,}")


def reset_lorma():
    for m in LORMA_MODULES:
        m.reset_parameters()

## 7. Chunk-level encoder and document head

In [ ]:
class DocHead(nn.Module):
    # Chunk vectors -> BiGRU -> attention pooling over chunks -> one logit per statute.
    def __init__(self, in_dim, hidden, n_out, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(in_dim, hidden, batch_first=True, bidirectional=True)
        self.att = nn.Linear(2 * hidden, 1)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * hidden, n_out)

    def forward(self, chunk_vecs, chunk_mask):
        h, _ = self.gru(chunk_vecs)
        scores = self.att(h).squeeze(-1)
        scores = scores.masked_fill(chunk_mask == 0, -1e4)
        weights = torch.softmax(scores, dim=1)
        doc = (weights.unsqueeze(-1) * h).sum(dim=1)
        return self.out(self.drop(doc))


HEAD = DocHead(HIDDEN + 1, HEAD_HIDDEN, N_STATUTES).to(DEVICE)
print("head params:", f"{sum(p.numel() for p in HEAD.parameters()):,}")


def pooled_backbone(ids_t, mask_t):
    # Backbone forward, then mean pooling over real (non-pad) tokens. Returns [N, HIDDEN].
    with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        hidden = backbone_hidden(ids_t, mask_t)
    hidden = hidden.float()
    m = mask_t.unsqueeze(-1).float()
    return (hidden * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)


def encode_chunks(chunk_list, rel_list):
    # chunk_list: list of id lists (each ends with SEP). Returns [N, HIDDEN + 1] (mean pool + position).
    ids, mask, lengths = lsi_utils.pad_right(chunk_list, PAD_ID)
    ids_t = torch.tensor(ids, device=DEVICE)
    mask_t = torch.tensor(mask, device=DEVICE)
    if torch.is_grad_enabled():
        # Gradient checkpointing: this micro-batch's activations are recomputed during backward
        # instead of being kept in memory. Needed on a 16 GB T4 with 32 chunks per document.
        mean_vec = torch_checkpoint(pooled_backbone, ids_t, mask_t, use_reentrant=False)
    else:
        mean_vec = pooled_backbone(ids_t, mask_t)
    rel = torch.tensor(rel_list, device=DEVICE, dtype=torch.float32).unsqueeze(-1)
    return torch.cat([mean_vec, rel], dim=-1)


def build_doc_features(ids_list, max_chunks):
    # ids_list: list of documents (each a numpy array of token ids).
    # Returns padded chunk features [B, C, HIDDEN+1] and chunk mask [B, C].
    flat_chunks = []
    flat_rel = []
    counts = []
    for ids in ids_list:
        # training and inference use the same chunking (same cap, same pick)
        chunks, rel = lsi_utils.make_chunks(ids, SEP_ID, CHUNK_LEN, CHUNK_OVERLAP, max_chunks, CHUNK_PICK)
        counts.append(len(chunks))
        flat_chunks.extend(chunks)
        flat_rel.extend(rel)

    pieces = []
    for start in range(0, len(flat_chunks), MICRO_CHUNKS):
        pieces.append(encode_chunks(flat_chunks[start:start + MICRO_CHUNKS], flat_rel[start:start + MICRO_CHUNKS]))
    flat_vec = torch.cat(pieces, dim=0)

    c_max = max(counts)
    batch = len(ids_list)
    padded = torch.zeros(batch, c_max, HIDDEN + 1, device=DEVICE, dtype=flat_vec.dtype)
    cmask = torch.zeros(batch, c_max, device=DEVICE)
    pos = 0
    for i in range(batch):
        c = counts[i]
        padded[i, :c] = flat_vec[pos:pos + c]
        cmask[i, :c] = 1.0
        pos += c
    return padded, cmask


# Gradient check: one backward pass through the checkpointed MoE backbone. B has a gradient at init
# (A has none until B moves), so every kind of target layer must show a nonzero B gradient.
for m in LORMA_MODULES:
    m.train()
HEAD.train()
grad_doc = np.random.randint(3, old_emb.num_embeddings, size=3000).astype(np.int64)
with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
    g_feats, g_mask = build_doc_features([grad_doc], MAX_CHUNKS_TRAIN)
    g_logits = HEAD(g_feats.float(), g_mask)
    g_loss = g_logits.float().sum()
g_loss.backward()
grad_by_kind = {}
for name, m in zip(LORMA_TARGETS, LORMA_MODULES):
    kind = name.split(".")[-1]
    if kind not in grad_by_kind:
        grad_by_kind[kind] = 0
    if m.lora_B.grad is not None and float(m.lora_B.grad.abs().max()) > 0:
        grad_by_kind[kind] += 1
for kind in grad_by_kind:
    print("  ", kind.ljust(10), "adapters with a gradient:", grad_by_kind[kind], "of", calls_by_kind[kind][0])
    assert grad_by_kind[kind] > 0, "no gradient reaches the adapters on " + kind
for p in LORMA_PARAMS + list(HEAD.parameters()):
    p.grad = None
for m in LORMA_MODULES:
    m.calls = 0

## 8. Loss, metrics, and checkpoints

In [ ]:
def make_pos_weight():
    raw = DATA["pos_weight"]
    values = []
    for s in STATUTES:
        w = float(raw.get(s, raw.get(str(s), 1.0)))
        if POS_WEIGHT_MODE == "sqrt":
            w = math.sqrt(w)
        values.append(w)
    return torch.tensor(values, dtype=torch.float32, device=DEVICE)


POS_WEIGHT = make_pos_weight()
LOSS_FN = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
print("pos_weight mode:", POS_WEIGHT_MODE, "| min:", round(float(POS_WEIGHT.min()), 2),
      "| max:", round(float(POS_WEIGHT.max()), 2))


def time_left_sec():
    return SESSION_CAP_SEC - (time.time() - START_TIME)


def ckpt_path(fold):
    return os.path.join(OUT_DIR, "ckpt_fold%d.pt" % fold)


def save_ckpt(fold, epoch_done, optimizer, scheduler, scaler):
    state = {
        "fold": fold,
        "epoch_done": epoch_done,
        "lora": [(m.lora_A.detach().cpu(), m.lora_B.detach().cpu()) for m in LORMA_MODULES],
        "head": HEAD.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict() if scaler is not None else None,
    }
    torch.save(state, ckpt_path(fold))
    print("checkpoint saved: fold", fold, "| epochs done:", epoch_done)


def load_ckpt(fold, optimizer, scheduler, scaler):
    if not os.path.exists(ckpt_path(fold)):
        return 0
    state = torch.load(ckpt_path(fold), map_location="cpu")
    for m, (a, b) in zip(LORMA_MODULES, state["lora"]):
        m.lora_A.data.copy_(a.to(DEVICE))
        m.lora_B.data.copy_(b.to(DEVICE))
    HEAD.load_state_dict(state["head"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    if scaler is not None and state["scaler"] is not None:
        scaler.load_state_dict(state["scaler"])
    print("resumed fold", fold, "from epoch", state["epoch_done"])
    return int(state["epoch_done"])


def best_label_thresholds(P, Y, global_t, min_pos):
    # One threshold per statute: the one that maximises that statute's F1.
    # Statutes with fewer than min_pos positives keep the global threshold.
    grid = np.arange(0.05, 0.951, 0.01)
    thr = np.full(P.shape[1], float(global_t))
    for j in range(P.shape[1]):
        pos = int(Y[:, j].sum())
        if pos < min_pos:
            continue
        best_f = -1.0
        best_t = float(global_t)
        for t in grid:
            pred = P[:, j] >= t
            tp = int((pred & (Y[:, j] == 1)).sum())
            fp = int((pred & (Y[:, j] == 0)).sum())
            fn = pos - tp
            denom = 2 * tp + fp + fn
            f = (2.0 * tp / denom) if denom > 0 else 0.0
            if f > best_f:
                best_f = f
                best_t = float(t)
        thr[j] = best_t
    return thr


def macro_micro_subset(Y_true, Y_pred):
    return lsi_utils.macro_micro_f1(Y_true, Y_pred)


@torch.no_grad()
def predict_probs(ids_all, max_chunks):
    LORMA_MODULES_EVAL = LORMA_MODULES
    for m in LORMA_MODULES_EVAL:
        m.eval()
    HEAD.eval()
    out = []
    for start in range(0, len(ids_all), EVAL_BATCH):
        docs = ids_all[start:start + EVAL_BATCH]
        feats, cmask = build_doc_features(docs, max_chunks)
        logits = HEAD(feats.float(), cmask)
        out.append(torch.sigmoid(logits).cpu().numpy())
    for m in LORMA_MODULES_EVAL:
        m.train()
    return np.concatenate(out, axis=0)

## 9. Train one fold (resumable, with a session time cap)

In [ ]:
def train_one_fold(fold):
    train_rows = [i for i in range(N_POOL) if pool_fold[i] != fold]
    val_rows = [i for i in range(N_POOL) if pool_fold[i] == fold]
    if SMOKE_TEST:
        train_rows = train_rows[:SMOKE_DOCS]
        val_rows = val_rows[:SMOKE_DOCS]
    epochs = SMOKE_EPOCHS if SMOKE_TEST else EPOCHS

    reset_lorma()
    HEAD.apply(lambda m: m.reset_parameters() if hasattr(m, "reset_parameters") else None)

    params = LORMA_PARAMS + list(HEAD.parameters())
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

    steps_per_epoch = math.ceil(len(train_rows) / (MICRO_BATCH * GRAD_ACC))
    total_steps = max(1, steps_per_epoch * epochs)
    warmup = max(1, int(WARMUP_FRAC * total_steps))

    def lr_lambda(step):
        if step < warmup:
            return float(step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda") if USE_SCALER else None

    epoch_start = load_ckpt(fold, optimizer, scheduler, scaler)
    if EVAL_ONLY:
        if not os.path.exists(ckpt_path(fold)):
            raise RuntimeError("EVAL_ONLY is on, but no checkpoint for fold %d was found. Attach the earlier run's output "
                               "(a folder named like OUT_DIR with ckpt_fold%d.pt) or set RESUME_DIR." % (fold, fold))
        epochs = epoch_start          # no more training: predict with what was trained so far
        steps_done = int(scheduler.last_epoch)
        print("EVAL_ONLY: no training. Finished epochs in the checkpoint:", epoch_start,
              "| optimizer steps done:", steps_done, "of", steps_per_epoch, "per epoch",
              "(= %.2f epochs of training)" % (steps_done / max(1, steps_per_epoch)))
        if epoch_start == 0:
            print("NOTE: this checkpoint was saved in the middle of the first epoch (session cap), so the weights are only partly trained.")

    rng = np.random.RandomState(SEED + fold)
    for epoch in range(epoch_start, epochs):
        if RARE_OVERSAMPLE:
            w = DOC_WEIGHT[train_rows]
            w = w / w.sum()
            order = list(rng.choice(train_rows, size=len(train_rows), replace=True, p=w))
        else:
            order = list(rng.permutation(train_rows))
        HEAD.train()
        for m in LORMA_MODULES:
            m.train()
        running = 0.0
        n_seen = 0
        t0 = time.time()
        micro_steps = math.ceil(len(order) / MICRO_BATCH)
        optimizer.zero_grad()
        for micro in range(micro_steps):
            if time_left_sec() < 300:
                save_ckpt(fold, epoch, optimizer, scheduler, scaler)
                print("session time cap reached. Run this notebook again to resume.")
                return None
            rows = order[micro * MICRO_BATCH:(micro + 1) * MICRO_BATCH]
            ids_list = [pool_ids[i] for i in rows]
            y = torch.tensor(Y_POOL[rows], dtype=torch.float32, device=DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                feats, cmask = build_doc_features(ids_list, MAX_CHUNKS_TRAIN)
                logits = HEAD(feats.float(), cmask)
                loss = LOSS_FN(logits.float(), y) / GRAD_ACC
            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            running += float(loss.detach()) * GRAD_ACC * len(rows)
            n_seen += len(rows)

            last_micro = (micro == micro_steps - 1)
            if (micro + 1) % GRAD_ACC == 0 or last_micro:
                if scaler is not None:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            if micro % 500 == 0:
                print("epoch", epoch + 1, "| micro", micro, "/", micro_steps,
                      "| loss", round(running / max(1, n_seen), 5),
                      "| elapsed", round(time.time() - t0, 1), "s")

        print("epoch", epoch + 1, "done | train loss", round(running / max(1, n_seen), 5),
              "| seconds", round(time.time() - t0, 1))
        save_ckpt(fold, epoch + 1, optimizer, scheduler, scaler)

    # validation on the held-out fold, all chunks
    val_ids = [pool_ids[i] for i in val_rows]
    val_probs = predict_probs(val_ids, MAX_CHUNKS_EVAL)
    val_true = Y_POOL[val_rows]
    macro05, micro05, subset05 = macro_micro_subset(val_true, (val_probs >= 0.5).astype(int))
    best_t, best_macro = lsi_utils.best_global_threshold(val_probs, val_true)
    print("fold", fold, "| macro-F1 @0.5:", round(macro05, 4), "| micro-F1 @0.5:", round(micro05, 4),
          "| subset acc @0.5:", round(subset05, 4))
    print("fold", fold, "| best global threshold:", round(best_t, 2), "| macro-F1 at that threshold:", round(best_macro, 4))

    np.save(os.path.join(OUT_DIR, "oof_probs_fold%d.npy" % fold), val_probs.astype(np.float32))
    with open(os.path.join(OUT_DIR, "oof_doc_ids_fold%d.json" % fold), "w") as f:
        json.dump([pool_doc_ids[i] for i in val_rows], f)
    np.save(os.path.join(OUT_DIR, "oof_labels_fold%d.npy" % fold), val_true.astype(np.uint8))

    # test predictions for this fold's model
    test_probs = predict_probs(test_ids, MAX_CHUNKS_EVAL)
    np.save(os.path.join(OUT_DIR, "test_probs_fold%d.npy" % fold), test_probs.astype(np.float32))
    print("saved OOF and test probabilities for fold", fold)

    return {"fold": fold, "macro_f1_0.5": macro05, "micro_f1_0.5": micro05,
            "subset_acc_0.5": subset05, "best_threshold": best_t, "best_macro_f1": best_macro,
            "epochs_trained": epochs}


RESULTS = []
for fold in RUN_FOLDS:
    print("\n=== fold", fold, "===")
    result = train_one_fold(fold)
    if result is None:
        break
    RESULTS.append(result)

print("\nfolds finished:", [r["fold"] for r in RESULTS])

## 10. Combine folds, pick a threshold, write the submission

In [ ]:
done_folds = [r["fold"] for r in RESULTS]
if len(done_folds) > 0:
    # threshold chosen on the out-of-fold probabilities of the folds that finished
    oof_list, label_list = [], []
    for f in done_folds:
        oof_list.append(np.load(os.path.join(OUT_DIR, "oof_probs_fold%d.npy" % f)))
        label_list.append(np.load(os.path.join(OUT_DIR, "oof_labels_fold%d.npy" % f)))
    oof_probs = np.concatenate(oof_list, axis=0)
    oof_true = np.concatenate(label_list, axis=0).astype(np.int64)
    global_t, global_macro = lsi_utils.best_global_threshold(oof_probs, oof_true)
    print("OOF rows:", oof_probs.shape[0], "| global threshold:", round(global_t, 2),
          "| OOF macro-F1:", round(global_macro, 4))

    # Honest check: tune per-label thresholds on half the OOF rows, score on the other half.
    half = (np.arange(oof_probs.shape[0]) % 2) == 0
    thr_a = best_label_thresholds(oof_probs[half], oof_true[half], global_t, THRESH_MIN_POS)
    thr_b = best_label_thresholds(oof_probs[~half], oof_true[~half], global_t, THRESH_MIN_POS)
    glob_on_b = lsi_utils.macro_micro_f1(oof_true[~half], (oof_probs[~half] >= global_t).astype(int))[0]
    per_on_b = lsi_utils.macro_micro_f1(oof_true[~half], (oof_probs[~half] >= thr_a).astype(int))[0]
    glob_on_a = lsi_utils.macro_micro_f1(oof_true[half], (oof_probs[half] >= global_t).astype(int))[0]
    per_on_a = lsi_utils.macro_micro_f1(oof_true[half], (oof_probs[half] >= thr_b).astype(int))[0]
    print("held-out check | global threshold macro-F1:", round((glob_on_a + glob_on_b) / 2, 4),
          "| per-label thresholds macro-F1:", round((per_on_a + per_on_b) / 2, 4))

    # final thresholds use all OOF rows
    thr_all = best_label_thresholds(oof_probs, oof_true, global_t, THRESH_MIN_POS)
    final_pred = (oof_probs >= thr_all).astype(int)
    final_macro, final_micro, final_subset = lsi_utils.macro_micro_f1(oof_true, final_pred)
    print("final per-label thresholds on all OOF | macro-F1:", round(final_macro, 4),
          "| micro-F1:", round(final_micro, 4), "| subset acc:", round(final_subset, 4))
    with open(os.path.join(OUT_DIR, "per_label_thresholds.json"), "w") as f:
        json.dump({str(STATUTES[j]): float(thr_all[j]) for j in range(N_STATUTES)}, f, indent=1)

    test_avg = np.mean([np.load(os.path.join(OUT_DIR, "test_probs_fold%d.npy" % f)) for f in done_folds], axis=0)

    pred_lists = []
    for i in range(test_avg.shape[0]):
        row = test_avg[i]
        chosen = [STATUTES[j] for j in range(N_STATUTES) if row[j] >= thr_all[j]]
        if len(chosen) == 0:
            # every training document has at least one statute, so always predict the top one
            chosen = [STATUTES[int(np.argmax(row))]]
        pred_lists.append(chosen)

    sub_path = os.path.join(OUT_DIR, "submission_lsi_m4_lorma.json")
    lsi_utils.write_submission(test_doc_ids, pred_lists, STATUTES, sub_path)

    summary = {
        "model": "Phase-1 Model 4 (GQA + MHA, MoE) + LoRMA",
        "lorma": {"r": LORMA_R, "alpha": LORMA_ALPHA, "dropout": LORMA_DROPOUT, "targets": len(LORMA_TARGETS),
                 "target_names": LORMA_TARGET_NAMES},
        "folds": RESULTS,
        "oof_macro_f1_global_threshold": global_macro,
        "global_threshold": global_t,
        "oof_macro_f1_per_label_thresholds": final_macro,
        "held_out_check": {"global": (glob_on_a + glob_on_b) / 2, "per_label": (per_on_a + per_on_b) / 2},
        "config": {"chunk_len": CHUNK_LEN, "overlap": CHUNK_OVERLAP,
                   "max_chunks_train": MAX_CHUNKS_TRAIN, "max_chunks_eval": MAX_CHUNKS_EVAL,
                   "lr": LR, "epochs": EPOCHS, "pos_weight_mode": POS_WEIGHT_MODE,
                   "effective_batch": MICRO_BATCH * GRAD_ACC, "seed": SEED, "eval_only": EVAL_ONLY},
    }
    with open(os.path.join(OUT_DIR, "run_summary.json"), "w") as f:
        json.dump(summary, f, indent=1)
    print("run summary saved")
else:
    print("No fold finished yet. Run again to resume.")

print("elapsed (min):", round((time.time() - START_TIME) / 60, 1))

## 11. Check the files

In [ ]:
for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print(name.ljust(32), round(os.path.getsize(full) / 1e6, 2), "MB")

if len(RESULTS) > 0:
    sub = json.load(open(os.path.join(OUT_DIR, "submission_lsi_m4_lorma.json")))
    assert len(sub) == len(test_doc_ids)
    for row in sub[:50]:
        assert len(row["pred"]) == len(set(row["pred"])) > 0
        assert all(int(p) in set(STATUTES) for p in row["pred"])
    print("submission rows:", len(sub), "| checks passed")
print("Done.")

## Notes
- LoRMA targets the 7 named linear layers inside every transformer block (attention q,k,v,o and the feed-forward gate, up, down, which sit inside each expert in the MoE blocks). The router, embeddings and LM head are not adapted.
- Hidden states use the final layer norm (`ln_f`) of the Phase-1 model. Padded positions are routed through the experts like real tokens, as in the Phase-3 Model 4 runs.
- Chunking: up to 32 chunks per document for training and for inference (same pick), so the model sees at training time what it sees at test time.
- The first run should be `SMOKE_TEST = True` on Kaggle. Then time one epoch and set `EPOCHS` and `RUN_FOLDS`.
- The activity and gradient checks near the top print per-layer-kind counts. If one of them fails, the MoE code bypasses the wrapped layers and the adapter placement must be changed.
- Set `MICRO_CHUNKS = 2` or `MICRO_BATCH = 1` if the GPU runs out of memory.
- Resume: a new Kaggle run has an empty working folder. Attach the earlier run's output (or a Dataset made from it) as an input; the folder with the same name as `OUT_DIR` is found and its checkpoints are copied back. `EVAL_ONLY = True` then skips training and only predicts from the checkpoint.